# Titanic Feature Engineering Pipeline

Titanic Dataset을 이용해 EDA, Feature Engineering, Feature Selection, 모델 비교, GridSearchCV 예시를 수행하는 과제용 노트북입니다. Colab에서는 먼저 `titanic.csv`를 업로드하거나 코드의 URL 다운로드 부분을 사용하면 됩니다.

In [ ]:
import os, json, warnings
warnings.filterwarnings('ignore')
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler, MinMaxScaler, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,roc_auc_score
OUT='./titanic_outputs'; os.makedirs(OUT, exist_ok=True)
df_raw=pd.read_csv('titanic.csv') if os.path.exists('titanic.csv') else pd.read_csv('https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv')
def add_features(df):
    df=df.copy(); df['FamilySize']=df.SibSp+df.Parch+1; df['IsAlone']=(df.FamilySize==1).astype(int); df['FarePerPerson']=df.Fare/df.FamilySize.replace(0,1); df['HasCabin']=df.Cabin.notna().astype(int)
    title=df.Name.str.extract(r',\s*([^\.]+)\.', expand=False).replace(['Lady','Countess','Capt','Col','Don','Dr','Major','Rev','Sir','Jonkheer','Dona'],'Rare').replace(['Mlle','Ms'],'Miss').replace('Mme','Mrs'); df['Title']=title
    df['AgeGroup']=pd.cut(df.Age,[0,12,18,35,60,120],labels=['Child','Teen','YoungAdult','Adult','Senior']).astype('object').fillna('Unknown')
    return df
df=add_features(df_raw)
# plots
sns.set_theme(style='whitegrid')
plots=[('hist_age', lambda: sns.histplot(df_raw['Age'].dropna(),bins=30,kde=True,color='steelblue'), 'Age Distribution'),('box_fare_survived', lambda: sns.boxplot(data=df_raw,x='Survived',y='Fare',color='lightblue'), 'Fare by Survival'),('count_survived', lambda: sns.countplot(data=df_raw,x='Survived',color='steelblue'), 'Target Distribution'),('bar_sex_survival', lambda: sns.barplot(data=df_raw,x='Sex',y='Survived',color='steelblue'), 'Survival Rate by Sex')]
for name,func,title in plots:
    plt.figure(figsize=(6,3.5)); func(); plt.title(title); plt.tight_layout(); plt.savefig(f'{OUT}/{name}.png',dpi=150); plt.close()
plt.figure(figsize=(7,5)); cols=['Survived','Pclass','Age','SibSp','Parch','Fare','FamilySize','IsAlone','FarePerPerson','HasCabin']; sns.heatmap(df[cols].corr(numeric_only=True),annot=True,cmap='Blues',fmt='.2f',cbar=False); plt.title('Correlation Heatmap'); plt.tight_layout(); plt.savefig(f'{OUT}/heatmap_corr.png',dpi=150); plt.close()
# ML
selected=['Pclass','Sex','Age','SibSp','Parch','Fare','Embarked','FamilySize','IsAlone','FarePerPerson','HasCabin','Title','AgeGroup']; y=df.Survived; X=df[selected]
base=df[selected+['Survived']].dropna(subset=['Age','Embarked']); Xbase=base[selected]; ybase=base.Survived
cat=['Pclass','Sex','Embarked','HasCabin','Title','AgeGroup','IsAlone']; num=['Age','SibSp','Parch','Fare','FamilySize','FarePerPerson']
def ohe():
    try: return OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    except TypeError: return OneHotEncoder(handle_unknown='ignore', sparse=False)
def pre(num_strategy, encoding, scaler):
    sc={'standard':StandardScaler(),'minmax':MinMaxScaler(),'robust':RobustScaler(),None:'passthrough'}[scaler]
    enc=ohe() if encoding=='onehot' else OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    return ColumnTransformer([('num',Pipeline([('imputer',SimpleImputer(strategy=num_strategy)),('scaler',sc)]),num),('cat',Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),('encoder',enc)]),cat)])
exps=[('Base','Drop NA','ordinal',None,False,True,'median'),('Exp-1','mean','onehot','standard',False,False,'mean'),('Exp-2','median','ordinal','minmax',True,False,'median'),('Exp-3','most_frequent','onehot','robust',True,False,'most_frequent')]
models={'Logistic Regression':LogisticRegression(max_iter=1000,random_state=42),'Random Forest':RandomForestClassifier(n_estimators=150,random_state=42,class_weight='balanced')}
rows=[]; saved={}
for ename,miss,enc,scaler,fs,drop,strat in exps:
    Xu,yu=(Xbase,ybase) if drop else (X,y)
    Xtr,Xte,ytr,yte=train_test_split(Xu,yu,test_size=.2,random_state=42,stratify=yu)
    for mn,m in models.items():
        pipe=Pipeline([('preprocess',pre(strat,enc,scaler)),('feature_select',SelectKBest(mutual_info_classif,k=10) if fs else 'passthrough'),('model',m)])
        pipe.fit(Xtr,ytr); pred=pipe.predict(Xte); proba=pipe.predict_proba(Xte)[:,1]
        rows.append({'Experiment':ename,'Model':mn,'Missing':miss,'Encoding':enc,'Scaling':scaler or 'None','Feature Selection':'O' if fs else 'X','Accuracy':accuracy_score(yte,pred),'Precision':precision_score(yte,pred),'Recall':recall_score(yte,pred),'F1':f1_score(yte,pred),'ROC-AUC':roc_auc_score(yte,proba)})
        saved[(ename,mn)]=(pipe,Xtr,Xte,ytr,yte)
res=pd.DataFrame(rows); res[['Accuracy','Precision','Recall','F1','ROC-AUC']]=res[['Accuracy','Precision','Recall','F1','ROC-AUC']].round(4); res.to_csv(f'{OUT}/performance_results.csv',index=False)
pipe=saved[('Exp-3','Random Forest')][0]; names=pipe.named_steps['preprocess'].get_feature_names_out(); sel=pipe.named_steps['feature_select']; names=names[sel.get_support()]
fi=pd.DataFrame({'Feature':names,'Importance':pipe.named_steps['model'].feature_importances_}).sort_values('Importance',ascending=False); fi.to_csv(f'{OUT}/feature_importance.csv',index=False)
plt.figure(figsize=(7,4)); sns.barplot(data=fi.head(10),y='Feature',x='Importance',color='steelblue'); plt.title('Top Feature Importances'); plt.tight_layout(); plt.savefig(f'{OUT}/feature_importance.png',dpi=150); plt.close()
# FS compare
fsrows=[]
for fs in [False, True]:
    Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.2,random_state=42,stratify=y)
    pipe=Pipeline([('preprocess',pre('most_frequent','onehot','robust')),('feature_select',SelectKBest(mutual_info_classif,k=10) if fs else 'passthrough'),('model',RandomForestClassifier(n_estimators=150,random_state=42,class_weight='balanced'))])
    pipe.fit(Xtr,ytr); pred=pipe.predict(Xte); proba=pipe.predict_proba(Xte)[:,1]
    fsrows.append({'Feature Selection':'O' if fs else 'X','Accuracy':accuracy_score(yte,pred),'F1':f1_score(yte,pred),'ROC-AUC':roc_auc_score(yte,proba)})
fsdf=pd.DataFrame(fsrows).round(4); fsdf.to_csv(f'{OUT}/feature_selection_comparison.csv',index=False)
summary={'shape':df_raw.shape,'missing':(df_raw.isna().mean()*100).round(2).to_dict(),'target_counts':df_raw.Survived.value_counts().to_dict(),'results':res.to_dict('records'),'best':res.sort_values(['F1','ROC-AUC'],ascending=False).iloc[0].to_dict(),'fs_compare':fsdf.to_dict('records'),'top_features':fi.head(10).to_dict('records')}
open(f'{OUT}/summary.json','w',encoding='utf-8').write(json.dumps(summary,ensure_ascii=False,indent=2))
print(res.to_string(index=False)); print('BEST:', summary['best'])
